In [0]:
from pyspark.sql.functions import col, from_json, explode, current_date, date_sub
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from delta.tables import DeltaTable

# Filter for records created in the last 7 days (for weekly incremental processing)
raw_df = spark.read.table("workspace.fotmob.raw_sofascore_player_stats")
print(f"Processing {raw_df.count()} raw records from the last 7 days...")

# Define schema for sofascore player stats JSON
team_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True)
])

season_info_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("year", StringType(), True)
])

tournament_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("slug", StringType(), True)
])

# Schema only includes fields that ACTUALLY exist in Sofascore JSON

statistics_schema = StructType([
    StructField("goals", IntegerType(), True),
    StructField("assists", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("appearances", IntegerType(), True),
    StructField("minutesPlayed", IntegerType(), True),
    StructField("saves", IntegerType(), True),
    StructField("tackles", IntegerType(), True),
    StructField("keyPasses", IntegerType(), True),
    StructField("cleanSheet", IntegerType(), True),
    StructField("totalCross", IntegerType(), True),
    StructField("totalShots", IntegerType(), True),
    StructField("totalPasses", IntegerType(), True),
    StructField("accuratePasses", IntegerType(), True),
    StructField("accuratePassesPercentage", DoubleType(), True),
    StructField("redCards", IntegerType(), True),
    StructField("yellowCards", IntegerType(), True),
    StructField("blockedShots", IntegerType(), True),
    StructField("dribbledPast", IntegerType(), True),
    StructField("passToAssist", IntegerType(), True),
    StructField("expectedGoals", DoubleType(), True),
    StructField("expectedAssists", DoubleType(), True),
    StructField("goalsConceded", IntegerType(), True),
    StructField("interceptions", IntegerType(), True),
    StructField("shotsOnTarget", IntegerType(), True),
    StructField("aerialDuelsWon", IntegerType(), True),
    StructField("totalLongBalls", IntegerType(), True),
    StructField("accurateCrosses", IntegerType(), True),
    StructField("accurateCrossesPercentage", DoubleType(), True),
    StructField("accurateLongBalls", IntegerType(), True),
    StructField("accurateLongBallsPercentage", DoubleType(), True),
    StructField("errorLeadToGoal", IntegerType(), True),
    StructField("goalsAssistsSum", IntegerType(), True),
    StructField("bigChancesMissed", IntegerType(), True),
    StructField("bigChancesCreated", IntegerType(), True),
    StructField("outfielderBlocks", IntegerType(), True),
    StructField("successfulDribbles", IntegerType(), True),
    StructField("shotsFromInsideTheBox", IntegerType(), True),
    StructField("countRating", IntegerType(), True),
    StructField("totalRating", DoubleType(), True),
    # Goalkeeper-specific fields
    StructField("goalsPrevented", DoubleType(), True),
    StructField("savedShotsFromInsideTheBox", IntegerType(), True),
    StructField("penaltyFaced", IntegerType(), True),
    StructField("penaltySave", IntegerType(), True)
])

season_schema = StructType([
    StructField("team", team_schema, True),
    StructField("year", StringType(), True),
    StructField("season", season_info_schema, True),
    StructField("uniqueTournament", tournament_schema, True),
    StructField("statistics", statistics_schema, True)
])

full_schema = StructType([
    StructField("seasons", ArrayType(season_schema), True)
])

# Parse JSON and explode seasons array
df = raw_df.select(
    col("player_id"),
    col("createdate"),
    from_json(col("raw_json"), full_schema).alias("parsed")
).select(
    col("player_id"),
    col("createdate"),
    explode(col("parsed.seasons")).alias("season_data")
).filter(
    col("season_data.team.gender") == "F"  # Only process female players
).select(
    col("player_id"),
    col("createdate"),
    col("season_data.year").alias("season"),
    col("season_data.uniqueTournament.name").alias("competition"),
    col("season_data.team.name").alias("team"),
    col("season_data.statistics.*")  # Flatten all statistics fields
)

# Rename columns to match fotmob naming convention (snake_case)
# Only renaming fields that actually exist in Sofascore JSON
df = df \
    .withColumnRenamed("minutesPlayed", "minutes_played") \
    .withColumnRenamed("keyPasses", "key_passes") \
    .withColumnRenamed("cleanSheet", "clean_sheets") \
    .withColumnRenamed("totalCross", "total_cross") \
    .withColumnRenamed("totalShots", "total_shots") \
    .withColumnRenamed("totalPasses", "total_passes") \
    .withColumnRenamed("accuratePasses", "accurate_passes") \
    .withColumnRenamed("accuratePassesPercentage", "accurate_passes_percentage") \
    .withColumnRenamed("redCards", "red_cards") \
    .withColumnRenamed("yellowCards", "yellow_cards") \
    .withColumnRenamed("blockedShots", "blocked_shots") \
    .withColumnRenamed("dribbledPast", "dribbled_past") \
    .withColumnRenamed("passToAssist", "passes_to_assist") \
    .withColumnRenamed("expectedGoals", "expected_goals") \
    .withColumnRenamed("expectedAssists", "expected_assists") \
    .withColumnRenamed("goalsConceded", "goals_conceded") \
    .withColumnRenamed("shotsOnTarget", "shots_on_target") \
    .withColumnRenamed("aerialDuelsWon", "aerials_won") \
    .withColumnRenamed("totalLongBalls", "total_long_balls") \
    .withColumnRenamed("accurateCrosses", "accurate_crosses") \
    .withColumnRenamed("accurateCrossesPercentage", "accurate_crosses_percentage") \
    .withColumnRenamed("accurateLongBalls", "accurate_long_balls") \
    .withColumnRenamed("accurateLongBallsPercentage", "accurate_long_balls_percentage") \
    .withColumnRenamed("errorLeadToGoal", "error_lead_to_goal") \
    .withColumnRenamed("goalsAssistsSum", "goals_assists_sum") \
    .withColumnRenamed("bigChancesMissed", "big_chances_missed") \
    .withColumnRenamed("bigChancesCreated", "big_chances_created") \
    .withColumnRenamed("outfielderBlocks", "outfielder_blocks") \
    .withColumnRenamed("successfulDribbles", "successful_dribbles") \
    .withColumnRenamed("shotsFromInsideTheBox", "shots_inside_box") \
    .withColumnRenamed("countRating", "count_rating") \
    .withColumnRenamed("totalRating", "total_rating") \
    .withColumnRenamed("goalsPrevented", "goals_prevented") \
    .withColumnRenamed("savedShotsFromInsideTheBox", "saved_shots_inside_box") \
    .withColumnRenamed("penaltyFaced", "penalty_faced") \
    .withColumnRenamed("penaltySave", "penalty_save")

print(f"After filtering for female players: {df.count()} records")
display(df)

Processing 871 raw records from the last 7 days...
After filtering for female players: 3200 records


player_id,createdate,season,competition,team,goals,assists,rating,appearances,minutes_played,saves,tackles,key_passes,clean_sheets,total_cross,total_shots,total_passes,accurate_passes,accurate_passes_percentage,red_cards,yellow_cards,blocked_shots,dribbled_past,passes_to_assist,expected_goals,expected_assists,goals_conceded,interceptions,shots_on_target,aerials_won,total_long_balls,accurate_crosses,accurate_crosses_percentage,accurate_long_balls,accurate_long_balls_percentage,error_lead_to_goal,goals_assists_sum,big_chances_missed,big_chances_created,outfielder_blocks,successful_dribbles,shots_inside_box,count_rating,total_rating,goals_prevented,saved_shots_inside_box,penalty_faced,penalty_save
28482,2026-07-13,25/26,WE-League,Urawa Red Diamonds Ladies,0,0,6.5,2,50,0,0,1,0,1,0,15,12,80.0,0,0,0,0,0,0.0,0.03,0,1,0,0,0,1,100.0,0,null,0,0,0,0,0,0,0,2,13.0,null,null,null,null
28482,2026-07-13,23/24,WE-League,Urawa Red Diamonds Ladies,2,0,7.33,7,623,0,1,2,1,2,4,346,296,85.55,0,0,1,2,1,0.95,0.26,7,12,2,15,41,1,50.0,23,56.1,0,2,0,0,1,3,0,7,51.3,null,null,null,null
28580,2026-07-13,25/26,A-League Women,Melbourne City,0,0,6.9,1,90,5,0,0,0,null,null,41,27,65.85,0,0,null,0,0,null,null,1,0,null,null,18,null,null,5,27.78,0,0,0,0,null,0,0,1,6.9,null,3,null,null
28580,2026-07-13,25/26,AFC Women's Champions League,Melbourne City,0,0,null,2,134,0,0,0,2,null,null,null,null,null,0,0,null,0,0,null,null,0,0,null,null,null,null,null,null,null,0,0,0,0,null,0,0,null,null,null,null,null,null
28580,2026-07-13,2025,NPL Victoria Women,Boroondara Eagles FC,0,0,8.05,4,360,23,1,0,1,0,0,164,137,83.54,0,0,0,0,0,0.0,0.0,5,8,0,5,64,0,null,41,64.06,0,0,0,0,0,0,0,4,32.2,null,13,null,null
28580,2026-07-13,24/25,A-League Women,Melbourne City,0,0,6.53,3,118,2,0,0,0,null,null,41,34,82.93,0,0,null,0,0,null,null,1,0,null,null,14,null,null,8,57.14,1,0,0,0,null,0,0,3,19.6,null,2,null,null
28580,2026-07-13,24/25,AFC Women's Champions League,Melbourne City,0,0,null,1,90,0,0,0,1,null,null,null,null,null,0,0,null,0,0,null,null,0,0,null,null,null,null,null,null,null,0,0,0,0,null,0,0,null,null,null,null,null,null
28580,2026-07-13,23/24,A-League Women,Melbourne City,0,0,7.28,5,405,19,0,0,1,null,null,130,107,82.31,0,0,null,0,0,null,null,5,0,null,null,26,null,null,11,42.31,0,0,0,0,null,0,0,5,36.4,null,11,1,1
28580,2026-07-13,22/23,A-League Women,Melbourne City,0,0,6.92,10,688,25,0,0,3,null,null,274,215,78.47,0,0,null,0,0,null,null,11,0,null,null,103,null,null,49,47.57,0,0,0,0,null,0,0,10,69.2,null,14,1,0
28580,2026-07-13,21/22,A-League Women,Melbourne City,0,0,7.02,15,1353,45,0,1,4,null,null,414,269,64.98,1,1,null,2,0,null,null,15,0,null,6,249,null,null,108,43.37,0,0,0,1,null,0,0,15,105.3,null,29,2,0


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Deduplicate: keep only the most recent record for each player/season/competition
window_spec = Window.partitionBy("player_id", "season", "competition").orderBy(col("createdate").desc())

df_deduped = df.withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num", "createdate")

print(f"After deduplication: {df_deduped.count()} unique player/season/competition records")
display(df_deduped)

After deduplication: 3163 unique player/season/competition records


player_id,season,competition,team,goals,assists,rating,appearances,minutes_played,saves,tackles,key_passes,clean_sheets,total_cross,total_shots,total_passes,accurate_passes,accurate_passes_percentage,red_cards,yellow_cards,blocked_shots,dribbled_past,passes_to_assist,expected_goals,expected_assists,goals_conceded,interceptions,shots_on_target,aerials_won,total_long_balls,accurate_crosses,accurate_crosses_percentage,accurate_long_balls,accurate_long_balls_percentage,error_lead_to_goal,goals_assists_sum,big_chances_missed,big_chances_created,outfielder_blocks,successful_dribbles,shots_inside_box,count_rating,total_rating,goals_prevented,saved_shots_inside_box,penalty_faced,penalty_save
25746,2019,FIFA Women's World Cup,Norway,0,0,6.8,2,31,0,1,0,0,null,1,12,9,75.0,0,0,0,0,0,null,null,1,0,1,4,1,null,null,1,100.0,0,0,0,0,null,1,0,2,13.6,null,null,null,null
25746,2023,"Toppserien, Women",Vålerenga,6,1,7.15,21,1496,0,0,14,2,10,34,707,613,86.7,0,0,5,13,0,8.24,1.36,20,23,20,29,29,6,60.0,13,44.83,0,7,5,1,5,16,0,21,150.1,null,null,null,null
25746,2024,"Toppserien, Women",Vålerenga,8,2,7.23,20,1530,0,3,12,4,4,19,830,729,87.83,0,0,11,5,0,7.35,1.14,13,37,12,24,74,1,25.0,43,58.11,0,10,3,1,11,4,0,20,144.6,null,null,null,null
25746,2025,"Toppserien, Women",Vålerenga,8,2,7.15,21,1349,0,2,15,5,4,19,736,642,87.23,0,0,2,5,2,6.69,1.14,12,25,14,19,59,3,75.0,32,54.24,0,10,3,1,2,2,0,19,135.9,null,null,null,null
25746,24/25,UEFA Women's Champions League,Vålerenga,1,0,6.78,4,359,0,6,0,0,null,2,115,93,80.87,0,1,3,2,0,null,null,8,4,1,6,19,null,null,8,42.11,0,1,0,0,3,1,2,4,27.1,null,null,null,null
25746,25/26,UEFA Women's Champions League,Vålerenga,0,0,6.57,3,47,0,1,1,0,1,3,12,9,75.0,0,0,0,0,0,null,null,1,0,1,2,null,0,0.0,null,null,0,0,0,0,null,1,2,3,19.7,null,null,null,null
28478,23/24,WE-League,Chifure AS Elfen Saitama,0,0,5.9,1,8,0,0,0,0,0,1,1,1,100.0,0,0,0,0,0,0.6,0.0,0,0,0,0,0,0,null,0,null,0,0,1,0,0,0,0,1,5.9,null,null,null,null
28478,24/25,WE-League,Chifure AS Elfen Saitama,0,0,6.55,2,31,0,0,0,0,0,1,6,6,100.0,0,0,0,0,0,0.1,0.0,1,0,0,1,1,0,null,1,100.0,0,0,0,0,0,0,0,2,13.1,null,null,null,null
28478,25/26,WE-League,Chifure AS Elfen Saitama,0,0,6.55,4,39,0,0,0,0,0,0,10,6,60.0,0,0,0,0,0,0.0,0.0,0,0,0,1,1,0,null,1,100.0,0,0,0,0,0,0,0,2,13.1,null,null,null,null
28482,23/24,WE-League,Urawa Red Diamonds Ladies,2,0,7.33,7,623,0,1,2,1,2,4,346,296,85.55,0,0,1,2,1,0.95,0.26,7,12,2,15,41,1,50.0,23,56.1,0,2,0,0,1,3,0,7,51.3,null,null,null,null


In [0]:
from pyspark.sql.functions import when

# Create per90 columns for all numeric stats (excluding percentages, ratings, and counts)
# Formula: (stat / minutes_played) * 90

# Define which columns should get per90 versions
per90_columns = [
    'goals', 'assists', 'saves', 'tackles', 'key_passes', 'clean_sheets',
    'total_cross', 'total_shots', 'total_passes', 'accurate_passes',
    'red_cards', 'yellow_cards', 'blocked_shots', 'dribbled_past', 'passes_to_assist',
    'expected_goals', 'expected_assists', 'goals_conceded', 'interceptions', 'shots_on_target',
    'aerials_won', 'total_long_balls', 'accurate_crosses', 'accurate_long_balls',
    'error_lead_to_goal', 'goals_assists_sum', 'big_chances_missed', 'big_chances_created',
    'outfielder_blocks', 'successful_dribbles', 'shots_inside_box',
    'goals_prevented', 'saved_shots_inside_box', 'penalty_faced', 'penalty_save'
]

# Add per90 columns
for stat_col in per90_columns:
    per90_col_name = f"{stat_col}_per90"
    df_deduped = df_deduped.withColumn(
        per90_col_name,
        when(col("minutes_played") > 0, (col(stat_col) / col("minutes_played")) * 90)
        .otherwise(None)
    )

print(f"Added per90 columns for {len(per90_columns)} stats")
print(f"Total columns now: {len(df_deduped.columns)}")
display(df_deduped.limit(5))

Added per90 columns for 35 stats
Total columns now: 82


player_id,season,competition,team,goals,assists,rating,appearances,minutes_played,saves,tackles,key_passes,clean_sheets,total_cross,total_shots,total_passes,accurate_passes,accurate_passes_percentage,red_cards,yellow_cards,blocked_shots,dribbled_past,passes_to_assist,expected_goals,expected_assists,goals_conceded,interceptions,shots_on_target,aerials_won,total_long_balls,accurate_crosses,accurate_crosses_percentage,accurate_long_balls,accurate_long_balls_percentage,error_lead_to_goal,goals_assists_sum,big_chances_missed,big_chances_created,outfielder_blocks,successful_dribbles,shots_inside_box,count_rating,total_rating,goals_prevented,saved_shots_inside_box,penalty_faced,penalty_save,goals_per90,assists_per90,saves_per90,tackles_per90,key_passes_per90,clean_sheets_per90,total_cross_per90,total_shots_per90,total_passes_per90,accurate_passes_per90,red_cards_per90,yellow_cards_per90,blocked_shots_per90,dribbled_past_per90,passes_to_assist_per90,expected_goals_per90,expected_assists_per90,goals_conceded_per90,interceptions_per90,shots_on_target_per90,aerials_won_per90,total_long_balls_per90,accurate_crosses_per90,accurate_long_balls_per90,error_lead_to_goal_per90,goals_assists_sum_per90,big_chances_missed_per90,big_chances_created_per90,outfielder_blocks_per90,successful_dribbles_per90,shots_inside_box_per90,goals_prevented_per90,saved_shots_inside_box_per90,penalty_faced_per90,penalty_save_per90
25746,2019,FIFA Women's World Cup,Norway,0,0,6.8,2,31,0,1,0,0,null,1,12,9,75.0,0,0,0,0,0,null,null,1,0,1,4,1,null,null,1,100.0,0,0,0,0,null,1,0,2,13.6,null,null,null,null,0.0,0.0,0.0,2.903225806451613,0.0,0.0,null,2.903225806451613,34.83870967741935,26.12903225806452,0.0,0.0,0.0,0.0,0.0,null,null,2.903225806451613,0.0,2.903225806451613,11.612903225806452,2.903225806451613,null,2.903225806451613,0.0,0.0,0.0,0.0,null,2.903225806451613,0.0,null,null,null,null
25746,2023,"Toppserien, Women",Vålerenga,6,1,7.15,21,1496,0,0,14,2,10,34,707,613,86.7,0,0,5,13,0,8.24,1.36,20,23,20,29,29,6,60.0,13,44.83,0,7,5,1,5,16,0,21,150.1,null,null,null,null,0.36096256684491984,0.06016042780748663,0.0,0.0,0.8422459893048128,0.12032085561497326,0.6016042780748663,2.0454545454545454,42.533422459893046,36.8783422459893,0.0,0.0,0.30080213903743314,0.7820855614973262,0.0,0.49572192513368984,0.08181818181818183,1.2032085561497325,1.3836898395721926,1.2032085561497325,1.7446524064171125,1.7446524064171125,0.36096256684491984,0.7820855614973262,0.0,0.4211229946524064,0.30080213903743314,0.06016042780748663,0.30080213903743314,0.9625668449197861,0.0,null,null,null,null
25746,2024,"Toppserien, Women",Vålerenga,8,2,7.23,20,1530,0,3,12,4,4,19,830,729,87.83,0,0,11,5,0,7.35,1.14,13,37,12,24,74,1,25.0,43,58.11,0,10,3,1,11,4,0,20,144.6,null,null,null,null,0.47058823529411764,0.11764705882352941,0.0,0.1764705882352941,0.7058823529411764,0.23529411764705882,0.23529411764705882,1.1176470588235294,48.8235294117647,42.88235294117647,0.0,0.0,0.6470588235294117,0.29411764705882354,0.0,0.43235294117647055,0.06705882352941175,0.7647058823529411,2.1764705882352944,0.7058823529411764,1.4117647058823528,4.352941176470589,0.058823529411764705,2.5294117647058822,0.0,0.5882352941176471,0.1764705882352941,0.058823529411764705,0.6470588235294117,0.23529411764705882,0.0,null,null,null,null
25746,2025,"Toppserien, Women",Vålerenga,8,2,7.15,21,1349,0,2,15,5,4,19,736,642,87.23,0,0,2,5,2,6.69,1.14,12,25,14,19,59,3,75.0,32,54.24,0,10,3,1,2,2,0,19,135.9,null,null,null,null,0.5337286879169755,0.13343217197924387,0.0,0.13343217197924387,1.000741289844329,0.3335804299481097,0.26686434395848774,1.267605633802817,49.10303928836174,42.83172720533729,0.0,0.0,0.13343217197924387,0.3335804299481097,0.13343217197924387,0.44633061527057083,0.07605633802816901,0.8005930318754633,1.6679021497405486,0.9340252038547071,1.267605633802817,3.9362490733876947,0.20014825796886582,2.134914751667902,0.0,0.6671608598962194,0.20014825796886582,0.06671608598962193,0.13343217197924387,0.13343217197924387,0.0,null,null,null,nul

In [0]:
# Save to processed table
table_name = "workspace.fotmob.sofascore_player_stats_processed"

# Check if we have any data to process
row_count = df_deduped.count()
if row_count == 0:
    print(f"No new records to process in the last 7 days. Skipping write to preserve existing data.")
else:
    # If table doesn't exist, create it
    if not spark.catalog.tableExists(table_name):
        df_deduped.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Created table {table_name} with {row_count} rows")
    else:
        # Table exists - always use MERGE to upsert records (composite key: player_id, season, competition)
        delta_table = DeltaTable.forName(spark, table_name)
        delta_table.alias("target").merge(
            df_deduped.alias("source"),
            "target.player_id = source.player_id AND target.season = source.season AND target.competition = source.competition"
        ).whenMatchedUpdateAll(
        ).whenNotMatchedInsertAll(
        ).execute()
        print(f"Merged {row_count} records into {table_name} (updated existing records or inserted new ones)")
    
    # Show final row count
    final_count = spark.read.table(table_name).count()
    print(f"Total rows in {table_name}: {final_count}")

Merged 3163 records into workspace.fotmob.sofascore_player_stats_processed (updated existing records or inserted new ones)
Total rows in workspace.fotmob.sofascore_player_stats_processed: 3163
